# Временная устойчивость, прошлые ЛИМС и предупреждения по сере
Эксперимент v2: прогноз на 6 часов, отдельные периоды настройки, калибровки интервалов и оценки. Train до начала 2024/2025, validation январь–июнь, calibration июль–сентябрь, evaluation октябрь–декабрь; разрывы 48 часов. Это ретроспективная проверка на ранее исследованных данных. Интервалы имеют номинальный уровень 90%; при временном дрейфе покрытие не гарантируется. Задержка ЛИМС 24/48 ч — допущение.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
HERE=Path.cwd().resolve()
EDA=HERE if HERE.name=='eda' else HERE/'eda'
RUN=EDA/'experiments'/'lims_walkforward_20260914_v2'
m=pd.read_csv(RUN/'metrics.csv')
p=pd.read_csv(RUN/'predictions.csv',parse_dates=['timestamp'])
display(m[['target','year','variant','n','mae','median_ae','coverage','interval_halfwidth']])

,target,year,variant,n,mae,median_ae,coverage,interval_halfwidth
0,Mg.Sulfur,2024,telemetry,100,1.505501,1.222919,0.940000,3.665988
1,Mg.Sulfur,2024,history24,100,1.457259,1.172621,0.950000,3.549043
2,Mg.Sulfur,2024,history48,100,1.478062,1.189883,0.950000,3.588533
3,Mg.Sulfur,2024,median,100,1.585000,1.200000,0.960000,4.100000
4,Mg.Sulfur,2024,last24,100,1.977000,1.550000,0.840000,3.700000
5,Mg.Sulfur,2024,last48,100,1.859000,1.400000,0.860000,3.300000
6,Mg.Sulfur,2025,telemetry,120,1.291502,1.169629,0.958333,3.345288
7,Mg.Sulfur,2025,history24,120,1.230917,1.115788,0.950000,3.307158
8,Mg.Sulfur,2025,history48,120,1.240983,1.119290,0.958333,3.345776
9,Mg.Sulfur,2025,median,120,1.290833,1.200000,0.958333,3.300000


In [2]:
fig=px.bar(m,x='variant',y='mae',color='variant',facet_row='target',facet_col='year',height=1000,title='MAE прогноза на 6 часов: режимы эксперимента и baseline')
fig.update_yaxes(matches=None)
fig.show()

In [3]:
fig=px.bar(m,x='variant',y='coverage',color='variant',facet_row='target',facet_col='year',height=1000,title='Фактическое покрытие 90% интервалов')
fig.add_hline(y=.9,line_dash='dash')
fig.show()
display(m[m.target=='Mg.Sulfur'][['year','variant','point_tp','point_fp','point_fn','point_recall','upper90_tp','upper90_fp','upper90_fn','upper90_recall']])

,year,variant,point_tp,point_fp,point_fn,point_recall,upper90_tp,upper90_fp,upper90_fn,upper90_recall
0,2024,telemetry,0.0,0.0,10.0,0.000000,10.0,90.0,0.0,1.000000
1,2024,history24,0.0,0.0,10.0,0.000000,10.0,90.0,0.0,1.000000
2,2024,history48,0.0,0.0,10.0,0.000000,10.0,90.0,0.0,1.000000
3,2024,median,0.0,0.0,10.0,0.000000,10.0,90.0,0.0,1.000000
4,2024,last24,1.0,8.0,9.0,0.100000,8.0,79.0,2.0,0.800000
5,2024,last48,0.0,6.0,10.0,0.000000,10.0,72.0,0.0,1.000000
6,2025,telemetry,0.0,0.0,27.0,0.000000,27.0,93.0,0.0,1.000000
7,2025,history24,0.0,0.0,27.0,0.000000,27.0,93.0,0.0,1.000000
8,2025,history48,0.0,0.0,27.0,0.000000,27.0,93.0,0.0,1.000000
9,2025,median,0.0,0.0,27.0,0.000000,27.0,93.0,0.0,1.000000


In [4]:
p['absolute_error']=(p.actual-p.prediction).abs()
p['month']=p.timestamp.dt.to_period('M').astype(str)
monthly=p.groupby(['target','month','variant'],as_index=False).absolute_error.mean()
fig=px.line(monthly,x='month',y='absolute_error',color='variant',facet_row='target',height=900,title='Средняя абсолютная ошибка по месяцам')
fig.update_yaxes(matches=None)
fig.show()

In [5]:
for year in sorted(p.year.unique()):
    g=p[(p.target=='Mg.Sulfur')&(p.year==year)&(p.variant=='history24')].sort_values('timestamp')
    fig=go.Figure()
    fig.add_scatter(x=g.timestamp,y=g.upper90,name='Верхняя граница',line=dict(width=0))
    fig.add_scatter(x=g.timestamp,y=g.lower90,name='Интервал 90%',fill='tonexty',line=dict(width=0))
    fig.add_scatter(x=g.timestamp,y=g.prediction,name='Прогноз')
    fig.add_scatter(x=g.timestamp,y=g.actual,name='ЛИМС',mode='markers')
    fig.add_hline(y=10,line_dash='dash',annotation_text='10 мг/кг')
    fig.update_layout(title=f'Сера {year}, h6, история ЛИМС с задержкой 24 ч',yaxis_title='мг/кг')
    fig.show()

## Проверка первой модели на 2026 годе
Низкая MAE первой модели h0 не означала обнаружения превышений: 4 TP, 6 FP, 32 FN, всего 36 фактических превышений среди 248 анализов. Recall = 11.1%. Эти числа нельзя интерпретировать как защиту качества.

Следующие шаги: проверка режима и причин ошибок, доступности тегов, уточнение задержек ЛИМС. Выбор модели по evaluation недопустим для заявления независимого качества. Ни один из этих экспериментов не оценивает причинный эффект уставок или безопасность пуска.